# P02 MCP の仕組み ― 電文を自分の手で送る

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

社内規程のサーバに、ホストが送るのと同じ JSON-RPC を自分で書いて送り、返事をそのまま見ます。版は 2026-07-28 (initialize の握手もセッション ID もありません)。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- このノートブックは Claude を呼びません。API キーが無くても全部動きます。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. このノートブックは Claude を呼びません (API キー不要)。ラボのモジュールを読めるようにする
from common.nb import safe
from common.kit import use_labs, utf8_console
use_labs(); utf8_console()

## 1. stdio で 1 行 1 通の JSON を送る
サーバを子プロセスで立て、server/discover → tools/list → tools/call → 資料 → 定型文 の順に送ります。どの要求も `params._meta` に版・名乗り・できることを持ちます。返事は画面用に縮めています。

In [ ]:
from run_protocol import amain as protocol
with safe():
    r = await protocol(["--part", "stdio"])

## 2. 旧来の initialize を送ると
1 本の接続の「時代」は最初の 1 通で決まります。initialize で始めると古い版 (2025-11-25) になり、2026-07-28 の接続では initialize が断られます。

In [ ]:
with safe():
    r = await protocol(["--part", "legacy"])

## 3. Streamable HTTP で送る
同じサーバを `127.0.0.1` の空きポートで立て、POST にヘッダー (MCP-Protocol-Version・Mcp-Method・Mcp-Name) を付けます。`ttlMs`・`cacheScope` はキャッシュの目安です。最後に `Client(url)` が実際に送った POST を数えます。

In [ ]:
with safe():
    r = await protocol(["--part", "http"])

## 4. 同じことを Client(kb.mcp) で
同じプロセスでつなぎます (子プロセスもポートも無し)。受け取る tools/list は、上で生で受け取ったものと同じ中身です。

In [ ]:
with safe():
    r = await protocol(["--part", "client"])